# Pennywise — guided expense analysis

This notebook explains the bundled synthetic dataset before opening the Streamlit app. Run cells from top to bottom. A row represents one student in one month. The six expense categories add up to the target `total_expenses` after invalid negative category amounts are repaired.

## 1. Load the data and inspect its shape

`shape` reports rows and columns. `info()` shows data types and missing values. `describe()` summarizes numerical columns.

In [ ]:
import pandas as pd
from data_utils import DEFAULT_DATA_PATH, CATEGORY_COLUMNS, clean_dataset, load_dataset, iqr_outlier_summary, make_model_frame

raw = load_dataset(DEFAULT_DATA_PATH)
print('Rows and columns:', raw.shape)
print('Students:', raw.student_id.nunique())
print('Distinct months:', raw.month.nunique())
raw.info()
raw.describe().round(2)

## 2. Clean invalid values

There are six negative category amounts in the sample file. Negative expense values are not meaningful here, so the cleaning function replaces them with the median of that category. It recalculates total expenses from the cleaned category columns. Positive IQR outliers are not automatically removed: they may be legitimate high expenses.

In [ ]:
cleaned, cleaning_report = clean_dataset(raw)
cleaning_report

In [ ]:
cleaned.head()

## 3. Explore spending

The histogram shows the distribution of monthly total expenses. A boxplot is useful for spotting unusually high or low records. The IQR rule flags observations for review; it does not prove that they are errors.

In [ ]:
import matplotlib.pyplot as plt

cleaned['total_expenses'].hist(bins=20)
plt.xlabel('Total expenses (₹)')
plt.ylabel('Number of student-month records')
plt.title('Distribution of monthly expenses')
plt.show()

In [ ]:
cleaned[CATEGORY_COLUMNS + ['total_expenses']].plot(kind='box', figsize=(11, 5), rot=35)
plt.ylabel('Amount (₹)')
plt.title('Expense category boxplots')
plt.tight_layout()
plt.show()

In [ ]:
iqr_outlier_summary(cleaned).round(2)

## 4. Understand the model inputs and target

The model estimates next-month total expenses from monthly income, planned budget, month number and the student's previous-month expenses. The first record for each student has no previous month and is excluded from model fitting. The target is the sum of six cleaned expense categories.

In [ ]:
frame = make_model_frame(cleaned)
print('Rows available for training/evaluation:', len(frame))
print('Model features:', ['monthly_income', 'budget', 'month_number', 'previous_month_expenses'])
print('Target:', 'total_expenses')
frame[['student_id', 'month', 'monthly_income', 'budget', 'month_number', 'previous_month_expenses', 'total_expenses']].head(10)

## 5. Train and evaluate

Run `python train_model.py` in the terminal to train and save the model. It uses a student-wise group split so a student's rows are not shared between training and testing. Review MAE, RMSE, R² and the mean baseline in `reports/model_metrics.json`. If the model does not beat the baseline, report that honestly; do not claim high accuracy.

In [ ]:
from train_model import train_and_save
model, metrics, cleaned_data, model_frame, test_result = train_and_save()
{key: metrics[key] for key in ['train_rows', 'test_rows', 'train_students', 'test_students', 'mae', 'rmse', 'r2', 'baseline_mae']}

## 6. Make a new prediction

The fitted model can receive a new row of input values. Changing these values makes a new prediction; it does not retrain the model. The Streamlit app provides interactive controls for the same four features.

In [ ]:
new_student_month = pd.DataFrame([{'monthly_income': 15000, 'budget': 12000, 'month_number': 6, 'previous_month_expenses': 7400}])
print('Estimated monthly expenses (₹):', round(float(model.predict(new_student_month)[0]), 2))